# Market Cluster Aggressiveness Analysis

This notebook builds an explainable market clustering framework for 2026 budget-setting aggressiveness using two dimensions only:

1. **Opportunity Score** (ability to scale efficiently)
2. **Market Gap** (urgency / under-allocation pressure)

No linear weighted combination is used. Markets are assigned into 3 aggressiveness groups based on a transparent 2D matrix.

## Inputs
- 2026 budget split and CPL Scenario 5 (provided in brief)
- 2025 market data from `pwc reports/outputs/python_output_all.csv`
- Opportunity scoring logic from `opportunity.py`

## Outputs
- Market-level table with gap and opportunity metrics
- 2D scatter with decision quadrants and cluster labels
- Cluster-level uplift guidance (`+40%`, `+50%`, `+60%`)
- Methodology-ready visuals for slides

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Force a renderer that does not depend on notebook MIME/nbformat integration.
pio.renderers.default = 'browser'

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'budget_setting' else Path.cwd().resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from opportunity import OPPORTUNITY_CONFIG, compute_headroom_scores

DATA_FILE = BASE_DIR / 'pwc reports' / 'outputs' / 'python_output_all.csv'

print(f'Base dir: {BASE_DIR}')
print(f'Data file exists: {DATA_FILE.exists()}')
print(f"Plotly renderer: {pio.renderers.default}")

Base dir: /home/ali/repos/porsche
Data file exists: True
Plotly renderer: browser


In [ ]:
# 2026 budget split from brief
budget_2026_raw = [
    ('PCGB', 944, 275000),
    ('PD', 1427, 260000),
    ('PKO', 713, 270000),
    ('POF', 1406, 152000),
    ('PIT', 1024, 145000),
    ('TRK', 269, 225000),
    ('PCL', 3068, 120000),
    ('PIB SPA', 2570, 80000),
    ('PNO', 2643, 70000),
    ('PIB POR', 597, 80000),
    ('PCH', 3301, 80000),
    ('PTW', 181, 230000),
    ('PCA', 2481, 70000),
    ('PPL', 2860, 70000),
    ('PJ', 1606, 60000),
    ('PBR', 1606, 60000),
    ('PCNA', 611, 245000),
]

budget_2026 = pd.DataFrame(budget_2026_raw, columns=['Market', 'CPL_Scenario_5_EUR', 'Budget_2026_EUR'])

# Normalize naming to align with historical data keys where needed
market_name_map = {
    'PIB SPA': 'PIB Spa',
    'PIB POR': 'PIB Por',
}
budget_2026['Market_Normalized'] = budget_2026['Market'].replace(market_name_map)

budget_2026

,Market,CPL_Scenario_5_EUR,Budget_2026_EUR,Market_Normalized
0,PCGB,944,275000,PCGB
1,PD,1427,260000,PD
2,PKO,713,270000,PKO
3,POF,1406,152000,POF
4,PIT,1024,145000,PIT
5,TRK,269,225000,TRK
6,PCL,3068,120000,PCL
7,PIB SPA,2570,80000,PIB Spa
8,PNO,2643,70000,PNO
9,PIB POR,597,80000,PIB Por


In [ ]:
def load_market_baseline(data_file: Path) -> pd.DataFrame:
    df = pd.read_csv(data_file)
    required_cols = ['Market', 'Media Spend', 'DCFS']
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns in source data: {missing}')

    df['Media Spend'] = pd.to_numeric(df['Media Spend'], errors='coerce')
    df['DCFS'] = pd.to_numeric(df['DCFS'], errors='coerce')

    baseline = (
        df.groupby('Market', dropna=False)
        .agg(
            spend_2025=('Media Spend', 'sum'),
            registrations_2025=('DCFS', 'sum'),
        )
        .reset_index()
    )
    return baseline


def compute_market_opportunity(df_source: pd.DataFrame) -> pd.DataFrame:
    config = dict(OPPORTUNITY_CONFIG)
    results, missing = compute_headroom_scores(df_source.copy(), config)
    if missing:
        raise ValueError(f'Opportunity scoring missing columns: {missing}')

    market_opp = (
        results.groupby('Market', dropna=False)
        .agg(
            opportunity_score=('opportunity_score', 'mean'),
            headroom_score=('headroom_score', 'mean'),
            scale_score=('scale_score', 'mean'),
            predictability_penalty=('predictability_penalty', 'mean'),
        )
        .reset_index()
    )
    return market_opp


def build_analysis_frame(data_file: Path, budget_df: pd.DataFrame) -> pd.DataFrame:
    src = pd.read_csv(data_file)
    baseline = load_market_baseline(data_file)
    market_opp = compute_market_opportunity(src)

    analysis = (
        budget_df
        .merge(baseline, left_on='Market_Normalized', right_on='Market', how='left', suffixes=('', '_baseline'))
        .merge(market_opp, left_on='Market_Normalized', right_on='Market', how='left', suffixes=('', '_opp'))
    )

    # Internal market gap proxy: registration share minus budget share.
    # Positive values suggest a market appears under-allocated vs 2025 registration contribution.
    analysis['budget_share'] = analysis['Budget_2026_EUR'] / analysis['Budget_2026_EUR'].sum()
    analysis['reg_share_2025'] = analysis['registrations_2025'] / analysis['registrations_2025'].sum()
    analysis['market_gap_raw'] = analysis['reg_share_2025'] - analysis['budget_share']

    # Percentile-based normalization keeps interpretation simple and avoids linear weighted blending.
    analysis['opportunity_pct'] = analysis['opportunity_score'].rank(pct=True) * 100
    analysis['market_gap_pct'] = analysis['market_gap_raw'].rank(pct=True) * 100

    return analysis


analysis_df = build_analysis_frame(DATA_FILE, budget_2026)
analysis_df[['Market', 'Market_Normalized', 'Budget_2026_EUR', 'registrations_2025', 'opportunity_score', 'market_gap_raw']].sort_values('Market').reset_index(drop=True)

,Market,Market_Normalized,Budget_2026_EUR,registrations_2025,opportunity_score,market_gap_raw
0,PBR,PBR,60000,NaN,NaN,NaN
1,PCA,PCA,70000,92.0,45.257188,-0.008697
2,PCGB,PCGB,275000,601.0,47.440903,0.016333
3,PCH,PCH,80000,41.0,53.179073,-0.023460
4,PCL,PCL,120000,99.0,36.714495,-0.027286
5,PCNA,PCNA,245000,561.0,73.367500,0.019940
6,PD,PD,260000,520.0,66.835807,0.005278
7,PIB POR,PIB Por,80000,33.0,49.782836,-0.025147
8,PIB SPA,PIB Spa,80000,187.0,56.463188,0.007315
9,PIT,PIT,145000,412.0,55.154711,0.028660


In [ ]:
def assign_aggressiveness_cluster(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out['has_required_data'] = out['opportunity_pct'].notna() & out['market_gap_pct'].notna()

    valid_idx = out.index[out['has_required_data']]
    out.loc[valid_idx, 'opp_band'] = pd.qcut(
        out.loc[valid_idx, 'opportunity_pct'], q=3, labels=['Low', 'Mid', 'High']
    ).astype(str)
    out.loc[valid_idx, 'gap_band'] = pd.qcut(
        out.loc[valid_idx, 'market_gap_pct'], q=3, labels=['Low', 'Mid', 'High']
    ).astype(str)

    # Explainable 2D matrix mapping (no weighted blending).
    matrix_to_uplift = {
        ('High', 'High'): '+60%',
        ('High', 'Mid'): '+60%',
        ('Mid', 'High'): '+60%',

        ('High', 'Low'): '+50%',
        ('Mid', 'Mid'): '+50%',
        ('Low', 'High'): '+50%',

        ('Mid', 'Low'): '+40%',
        ('Low', 'Mid'): '+40%',
        ('Low', 'Low'): '+40%',
    }

    out['aggressiveness_uplift'] = 'Data Gap'
    for idx in valid_idx:
        key = (out.at[idx, 'opp_band'], out.at[idx, 'gap_band'])
        out.at[idx, 'aggressiveness_uplift'] = matrix_to_uplift.get(key, 'Data Gap')

    out['cluster_label'] = out['aggressiveness_uplift'].map({
        '+60%': 'Cluster A - High aggressiveness',
        '+50%': 'Cluster B - Medium aggressiveness',
        '+40%': 'Cluster C - Baseline aggressiveness',
        'Data Gap': 'Cluster D - Data gap (manual review)',
    })

    return out


cluster_df = assign_aggressiveness_cluster(analysis_df)
cluster_df[['Market', 'opportunity_pct', 'market_gap_pct', 'opp_band', 'gap_band', 'aggressiveness_uplift', 'cluster_label']].sort_values(['aggressiveness_uplift', 'Market'], ascending=[False, True]).reset_index(drop=True)

,Market,opportunity_pct,market_gap_pct,opp_band,gap_band,aggressiveness_uplift,cluster_label
0,PBR,NaN,NaN,NaN,NaN,Data Gap,Cluster D - Data gap (manual review)
1,PJ,NaN,NaN,NaN,NaN,Data Gap,Cluster D - Data gap (manual review)
2,PCNA,100.000000,80.000000,High,High,+60%,Cluster A - High aggressiveness
3,PD,93.333333,60.000000,High,Mid,+60%,Cluster A - High aggressiveness
4,PIB SPA,73.333333,66.666667,High,Mid,+60%,Cluster A - High aggressiveness
5,PIT,60.000000,86.666667,Mid,High,+60%,Cluster A - High aggressiveness
6,PNO,80.000000,40.000000,High,Mid,+60%,Cluster A - High aggressiveness
7,TRK,86.666667,100.000000,High,High,+60%,Cluster A - High aggressiveness
8,PCGB,33.333333,73.333333,Low,High,+50%,Cluster B - Medium aggressiveness
9,POF,46.666667,46.666667,Mid,Mid,+50%,Cluster B - Medium aggressiveness


In [ ]:
# Main chart: 2D market positioning (export-first to avoid renderer hangs)
x_med = cluster_df['market_gap_pct'].median()
y_med = cluster_df['opportunity_pct'].median()

fig = px.scatter(
    cluster_df,
    x='market_gap_pct',
    y='opportunity_pct',
    color='aggressiveness_uplift',
    text='Market',
    size='Budget_2026_EUR',
    size_max=38,
    hover_data={
        'Budget_2026_EUR': ':,.0f',
        'CPL_Scenario_5_EUR': ':,.0f',
        'opportunity_score': ':.1f',
        'market_gap_raw': ':.4f',
        'opp_band': True,
        'gap_band': True,
    },
    color_discrete_map={'+60%': '#1b9e77', '+50%': '#d95f02', '+40%': '#7570b3', 'Data Gap': '#9CA3AF'},
    title='Market Clusters: Opportunity Score vs Market Gap',
)
fig.update_traces(textposition='top center')
fig.add_vline(x=x_med, line_dash='dash', line_color='gray')
fig.add_hline(y=y_med, line_dash='dash', line_color='gray')
fig.update_layout(
    xaxis_title='Market Gap Percentile (higher = more under-allocated vs 2025 registrations)',
    yaxis_title='Opportunity Score Percentile (higher = stronger scaling potential)',
    legend_title='Recommended uplift band',
    template='plotly_white',
)

out_html = BASE_DIR / 'budget_setting' / 'market_clusters_scatter.html'
fig.write_html(out_html)
print(f'Saved scatter plot HTML: {out_html}')

Saved scatter plot HTML: /home/ali/repos/porsche/budget_setting/market_clusters_scatter.html


In [ ]:
# 3x3 matrix view for slide appendix (export-first)
matrix = (
    cluster_df[cluster_df['has_required_data']]
    .pivot_table(
        index='opp_band',
        columns='gap_band',
        values='Market',
        aggfunc='count',
        fill_value=0,
    )
    .reindex(index=['High', 'Mid', 'Low'], columns=['Low', 'Mid', 'High'])
)

heatmap = px.imshow(
    matrix,
    text_auto=True,
    color_continuous_scale='Blues',
    title='3x3 Market Matrix (count of markets per cell)',
    labels={'x': 'Market Gap Band', 'y': 'Opportunity Band', 'color': 'Market count'},
)
heatmap.update_layout(template='plotly_white')
out_html = BASE_DIR / 'budget_setting' / 'market_clusters_matrix.html'
heatmap.write_html(out_html)
print(f'Saved matrix plot HTML: {out_html}')

matrix

Saved matrix plot HTML: /home/ali/repos/porsche/budget_setting/market_clusters_matrix.html


gap_band,Low,Mid,High
opp_band,,,
High,0,3,2
Mid,3,1,1
Low,2,1,2


In [ ]:
# Cluster summary for slide narrative
cluster_summary = (
    cluster_df
    .groupby(['aggressiveness_uplift', 'cluster_label'], dropna=False)
    .agg(
        markets=('Market', 'count'),
        budget_2026_total=('Budget_2026_EUR', 'sum'),
        avg_opportunity_score=('opportunity_score', 'mean'),
        avg_market_gap_raw=('market_gap_raw', 'mean'),
    )
    .reset_index()
    .sort_values('aggressiveness_uplift', ascending=False)
)

cluster_summary['budget_2026_total'] = cluster_summary['budget_2026_total'].round(0)
cluster_summary['avg_opportunity_score'] = cluster_summary['avg_opportunity_score'].round(1)
cluster_summary['avg_market_gap_raw'] = cluster_summary['avg_market_gap_raw'].round(4)
cluster_summary

,aggressiveness_uplift,cluster_label,markets,budget_2026_total,avg_opportunity_score,avg_market_gap_raw
3,Data Gap,Cluster D - Data gap (manual review),2,120000,NaN,NaN
2,+60%,Cluster A - High aggressiveness,6,1025000,61.9,0.0183
1,+50%,Cluster B - Medium aggressiveness,3,657000,43.6,0.0210
0,+40%,Cluster C - Baseline aggressiveness,6,690000,47.4,-0.0207


In [ ]:
# Final export table for reporting / slide support
final_table = cluster_df[[
    'Market',
    'Market_Normalized',
    'Budget_2026_EUR',
    'CPL_Scenario_5_EUR',
    'registrations_2025',
    'opportunity_score',
    'market_gap_raw',
    'opportunity_pct',
    'market_gap_pct',
    'opp_band',
    'gap_band',
    'aggressiveness_uplift',
    'cluster_label',
]].sort_values(['aggressiveness_uplift', 'opportunity_pct', 'market_gap_pct'], ascending=[False, False, False])

out_csv = BASE_DIR / 'budget_setting' / 'market_cluster_aggressiveness_output.csv'
final_table.to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')
final_table

Saved: /home/ali/repos/porsche/budget_setting/market_cluster_aggressiveness_output.csv


,Market,Market_Normalized,Budget_2026_EUR,CPL_Scenario_5_EUR,registrations_2025,opportunity_score,market_gap_raw,opportunity_pct,market_gap_pct,opp_band,gap_band,aggressiveness_uplift,cluster_label
14,PJ,PJ,60000,1606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Data Gap,Cluster D - Data gap (manual review)
15,PBR,PBR,60000,1606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Data Gap,Cluster D - Data gap (manual review)
16,PCNA,PCNA,245000,611,561.0,73.367500,0.019940,100.000000,80.000000,High,High,+60%,Cluster A - High aggressiveness
1,PD,PD,260000,1427,520.0,66.835807,0.005278,93.333333,60.000000,High,Mid,+60%,Cluster A - High aggressiveness
5,TRK,TRK,225000,269,720.0,61.611642,0.061482,86.666667,100.000000,High,High,+60%,Cluster A - High aggressiveness
8,PNO,PNO,70000,2643,72.0,57.916667,-0.012913,80.000000,40.000000,High,Mid,+60%,Cluster A - High aggressiveness
7,PIB SPA,PIB Spa,80000,2570,187.0,56.463188,0.007315,73.333333,66.666667,High,Mid,+60%,Cluster A - High aggressiveness
4,PIT,PIT,145000,1024,412.0,55.154711,0.028660,60.000000,86.666667,Mid,High,+60%,Cluster A - High aggressiveness
3,POF,POF,152000,1406,240.0,52.552083,-0.010405,46.666667,46.666667,Mid,Mid,+50%,Cluster B - Medium aggressiveness
0,PCGB,PCGB,275000,944,601.0,47.440903,0.016333,33.333333,73.333333,Low,High,+50%,Cluster B - Medium aggressiveness
